In [11]:
%load_ext autoreload
%autoreload

import pickle
import pandas as pd
import numpy as np
from collections import defaultdict
from constants import NetworkSettings, DataGenerationSettings
from methods.methods_est import PropensityScoreMatcher, LinearRegressionEstimator
from methods.visualization import ResultsPlotter

The autoreload extension is already loaded. To reload it, use:
  %reload_ext autoreload


In [18]:
np.random.normal(0.3)

-0.11072784800624175

### Analysis

In [5]:
network_settings = NetworkSettings()
dgp_settings = DataGenerationSettings()
ps_matcher = PropensityScoreMatcher()
lr_estimator = LinearRegressionEstimator()

In [6]:
network_configs = network_settings.networks_types
neighbour_influences = dgp_settings.n_influence_list
feature_adj_sets_ps = network_settings.feature_adj_sets_ps

In [34]:
for influence in neighbour_influences:
    file_path = f"/Users/polinarevina/Desktop/thesis_new/src/simulation_results/sim_covariate_block_random_covariate_block_influence{influence}.pkl"
    with open(file_path, "rb") as f:
        data = pickle.load(f)

    results_dict = defaultdict(lambda: {"pvalue": [], "coef": []})

    for i_sim in range(dgp_settings.n_sim):
        individ_data_sim = data[i_sim]["individ_data"]
        for feature_name, features_ps in feature_adj_sets_ps.items():
    
            individ_data_matched = ps_matcher.match(df=individ_data_sim,covariate_cols=features_ps)
            coef, pvalue = lr_estimator.calculate_results(individ_data=individ_data_matched)
            results_dict[feature_name]["pvalue"].append(pvalue)
            results_dict[feature_name]["coef"].append(coef)
    results[influence] = dict(results_dict)
    
    for feature_set, features in feature_adj_sets_ps.items():
        results_plotter = ResultsPlotter(
            estimated_effect=results_dict[feature_set]["pvalue"],
            true_effect=dgp_settings.treatment_effect_mean,
            network_type="cov_blcok",
            assignment_type="random",
            neighbour_influence=influence,
            treatment_mean=dgp_settings.treatment_effect_mean,
            additional_feature_names=features
        )
        results_plotter.calculate_results()
        results_df = results_plotter.create_results()
        results_plotter.create_plots()
        all_results_df = pd.concat([all_results_df, results_df], ignore_index=True)

In [37]:
all_results_df

,Network,Assignment,Influence,Treatment,Mean Estimated,Mean True,Perc Diff (%),Feature Names
0,cov_blcok,random,0.3,0.3,0.353,0.3,16.194,"[covariate_0, covariate_1]"
1,cov_blcok,random,0.3,0.3,0.357,0.3,17.399,"[covariate_0, covariate_1, treated_neighbors]"
2,cov_blcok,random,0.3,0.3,0.355,0.3,16.656,"[covariate_0, covariate_1, degree_centrality, ..."
3,cov_blcok,random,0.3,0.3,0.353,0.3,16.135,"[covariate_0, covariate_1, degree_centrality, ..."
4,cov_blcok,random,0.3,0.3,0.354,0.3,16.503,"[covariate_0, covariate_1, community]"
5,cov_blcok,random,0.3,0.3,0.357,0.3,17.301,"[covariate_0, covariate_1, emb_0, emb_1, emb_2..."
6,cov_blcok,random,0.3,0.3,0.353,0.3,16.194,"[covariate_0, covariate_1]"
7,cov_blcok,random,0.3,0.3,0.357,0.3,17.399,"[covariate_0, covariate_1, treated_neighbors]"
8,cov_blcok,random,0.3,0.3,0.355,0.3,16.656,"[covariate_0, covariate_1, degree_centrality, ..."
9,cov_blcok,random,0.3,0.3,0.353,0.3,16.135,"[covariate_0, covariate_1, degree_centrality, ..."


In [36]:
individ_data_sim

,outcome,covariate_0,covariate_1,group,node,degree_centrality,betweenness_centrality,community,treated_neighbors_count,treated_neighbors_exposure,...,emb_55,emb_56,emb_57,emb_58,emb_59,emb_60,emb_61,emb_62,emb_63,propensity_score
0,17.920498,0.204531,1.174796,1,0,0.343434,0.007691,0,21,0.617647,...,0.032681,-0.162498,0.011223,-0.129748,0.101098,0.254898,-0.119462,0.150969,-0.099225,0.949181
1,13.876358,-0.352865,0.521128,1,1,0.313131,0.005936,0,15,0.483871,...,-0.092924,-0.001412,0.063128,-0.065726,0.083292,0.198607,-0.091955,0.165667,-0.078308,0.981489
2,7.822374,0.094564,-0.536966,0,2,0.393939,0.007337,0,20,0.512821,...,-0.030579,0.025072,0.005366,-0.183027,-0.236606,-0.052083,-0.071993,0.201275,-0.004097,0.369962
3,7.080455,-0.055129,-0.084986,0,3,0.404040,0.014384,0,22,0.550000,...,-0.123016,-0.011247,0.115763,-0.078584,-0.047955,0.018023,-0.190426,0.064144,-0.075027,0.058885
4,4.713347,-0.219904,-1.032625,1,4,0.333333,0.005315,0,21,0.636364,...,0.265060,-0.106682,0.086437,0.091735,-0.138668,0.048615,-0.306749,-0.024420,0.090605,0.844388
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
95,9.437458,0.172006,-0.139639,1,95,0.444444,0.008628,0,24,0.545455,...,0.128124,0.105952,-0.043847,-0.194680,-0.034576,0.073992,-0.065702,0.138796,-0.064611,0.864042
96,1.971268,-0.577687,-1.490219,0,96,0.373737,0.004845,0,20,0.540541,...,0.159172,-0.156708,0.069880,0.074624,0.123352,0.072077,-0.054948,0.128525,0.035411,0.585691
97,1.943133,-0.832805,-0.442402,0,97,0.242424,0.010140,0,14,0.583333,...,0.053553,-0.138043,0.180257,0.109903,0.258867,-0.333308,0.234527,-0.138311,-0.325569,0.048715
98,20.518085,2.004605,1.117706,0,98,0.272727,0.009443,0,15,0.555556,...,0.045282,-0.087602,0.006465,-0.018277,-0.074547,-0.092534,-0.000991,-0.014253,-0.038808,0.019948


### Observational data analysis

In [8]:
lr_estimator = LinearRegressionEstimator()
results = defaultdict(lambda: defaultdict(dict))
all_results_df = pd.DataFrame()

In [9]:
for influence in neighbour_influences:
    file_path = f"/Users/polinarevina/Desktop/thesis_new/src/simulation_results/sim_covariate_block_individual_and_neighbors_covariate_block_influence{influence}_.pkl"
    with open(file_path, "rb") as f:
        data = pickle.load(f)

    results_dict = defaultdict(lambda: {"pvalue": [], "coef": []})

    for i_sim in range(dgp_settings.n_sim):
        individ_data_sim = data[i_sim]["individ_data"]
        for feature_name, features_ps in feature_adj_sets_ps.items():
    
            individ_data_matched = ps_matcher.match(df=individ_data_sim,covariate_cols=features_ps)
            coef, pvalue = lr_estimator.calculate_results(individ_data=individ_data_matched)
            results_dict[feature_name]["pvalue"].append(pvalue)
            results_dict[feature_name]["coef"].append(coef)
    results[influence] = dict(results_dict)
    
    for feature_set, features in feature_adj_sets_ps.items():
        results_plotter = ResultsPlotter(
            estimated_effect=results_dict[feature_set]["pvalue"],
            true_effect=dgp_settings.treatment_effect_mean,
            network_type="cov_blcok",
            assignment_type="random",
            neighbour_influence=influence,
            treatment_mean=dgp_settings.treatment_effect_mean,
            additional_feature_names=features
        )
        results_plotter.calculate_results()
        results_df = results_plotter.create_results()
        results_plotter.create_plots()
        all_results_df = pd.concat([all_results_df, results_df], ignore_index=True)

In [10]:
all_results_df

,Network,Assignment,Influence,Treatment,Mean Estimated,Mean True,Perc Diff (%),Feature Names
0,cov_blcok,random,0.3,0.3,0.108,0.3,94.378,"[covariate_0, covariate_1]"
1,cov_blcok,random,0.3,0.3,0.092,0.3,105.806,"[covariate_0, covariate_1, treated_neighbors_c..."
2,cov_blcok,random,0.3,0.3,0.093,0.3,105.085,"[covariate_0, covariate_1, degree_centrality, ..."
3,cov_blcok,random,0.3,0.3,0.104,0.3,96.799,"[covariate_0, covariate_1, degree_centrality, ..."
4,cov_blcok,random,0.3,0.3,0.096,0.3,103.166,"[covariate_0, covariate_1, community]"
5,cov_blcok,random,0.3,0.3,0.090,0.3,107.825,"[covariate_0, covariate_1, emb_0, emb_1, emb_2..."
6,cov_blcok,random,0.6,0.3,0.114,0.3,89.956,"[covariate_0, covariate_1]"
7,cov_blcok,random,0.6,0.3,0.101,0.3,98.980,"[covariate_0, covariate_1, treated_neighbors_c..."
8,cov_blcok,random,0.6,0.3,0.102,0.3,98.739,"[covariate_0, covariate_1, degree_centrality, ..."
9,cov_blcok,random,0.6,0.3,0.110,0.3,92.745,"[covariate_0, covariate_1, degree_centrality, ..."
